# Maximal Marginal Relevance (MMR) in RAG

Maximal Marginal Relevance (MMR) is a retrieval strategy used in **Retrieval-Augmented Generation (RAG)** to solve the **"redundancy problem."** While standard similarity search (Top-K) only looks for documents most similar to the query, MMR balances relevance to the query with diversity among the results.

---

## 1. Why do we need MMR?
In a standard vector search, the system retrieves the top 3$k$ chunks that are mathematically closest to your question.4 However, if your vector database contains three very similar paragraphs about the same fact, a standard search will likely return all three.

* **The Problem:** This "echo chamber" effect wastes the LLM's limited context window on redundant information and might miss other relevant but slightly less similar perspectives.
* **The Solution:** MMR "penalizes" documents that are too similar to those already selected for the final context, ensuring the LLM sees a broader range of information.6

---

## The Core Problem: Redundancy
In standard **Top-K Similarity Search**, the retriever pulls the chunks most mathematically similar to the query. If your database contains three nearly identical paragraphs about the same topic, the retriever will return all three.

* **Result:** The LLM receives repetitive information.
* **Consequence:** It wastes the context window and might miss a different, slightly less "similar" chunk that contains a vital alternative perspective.

---

## 2. The MMR Solution
MMR balances two competing goals:
1.  **Relevance:** How well the chunk answers the user's query.
2.  **Diversity:** How different the chunk is from the information already selected.

**It selects documents that are both highly relevant to the query and sufficiently different from already selected documents, reducing redundancy in retrieved context.**

### The Scoring Formula
MMR selects documents iteratively by maximizing the following score:

$$\text{MMR}(D_i) = \text{argmax}_{D_i \in R \setminus S} \left[ \lambda \cdot \text{Sim}(D_i, Q) - (1-\lambda) \cdot \max_{D_j \in S} \text{Sim}(D_i, D_j) \right]$$

* **$\text{Sim}(D_i, Q)$**: Similarity of candidate document to the query.
* **$\text{Sim}(D_i, D_j)$**: Similarity of the candidate to documents already in the selected set.
* **$\lambda$ (Lambda)**: The "diversity knob."


**Breakdown of terms:**
* **$\text{Sim}(D_i, Q)$:** How relevant the candidate is to your query.
* $\max_{D_j \in S} \text{Sim}(D_i, D_j)$: How similar the candidate is to the documents you have already picked.
* $\lambda$ (Lambda): The trade-off parameter (typically 0.5).

    * $\lambda = 1$: Pure similarity search (ignores diversity).

    * $\lambda = 0$: Pure diversity search (ignores relevance after the first pick)
---

## 3. How the Algorithm Works

MMR doesn't just rank everything once; it builds the final list iteratively:
1. Fetch Candidates: Retrieve a larger pool of documents (e.g., top 20) using standard similarity. This is often called fetch_k.
2.  First Pick: Select the document with the absolute highest similarity to the query.
3. Iterative Re-ranking: To pick the next document, calculate the MMR score for all remaining candidates.
4. Repeat: Pick the candidate with the highest MMR score, add it to the "selected" set, and repeat until you have 15$k$ documents.


# 4. MMR vs. Standard Retrieval: A Concrete Example

This example demonstrates how **Maximal Marginal Relevance (MMR)** improves the quality of a RAG system by preventing redundant information from crowding out useful context.

---

## 1. The Scenario
**User Query:** *"What are the best vacation spots in Europe?"*

Imagine your vector database contains the following four chunks, ranked by their mathematical similarity to the query:

| Chunk | Content | Similarity Score |
| :--- | :--- | :--- |
| **Chunk A** | "Paris is a top destination known for the Eiffel Tower and cafes." | 0.95 |
| **Chunk B** | "The Eiffel Tower in Paris is a must-see landmark for tourists." | 0.92 |
| **Chunk C** | "The Swiss Alps offer world-class skiing and hiking trails." | 0.85 |
| **Chunk D** | "Rome features historic sites like the Colosseum and great food." | 0.80 |

---

## 2. Standard Top-2 Retrieval
In a standard similarity search where $k=2$, the system simply picks the two highest-scoring documents.

* **Selected:** Chunk A and Chunk B.
* **Result:** The LLM receives two different paragraphs that both discuss the Eiffel Tower and Paris.
* **The Problem:** The LLM remains "blind" to the fact that the database contains information about Switzerland or Italy. The context is redundant.

---

## 3. Retrieval with MMR ($\lambda = 0.5$)
MMR evaluates documents one by one, factoring in how much **new** information they provide compared to what has already been picked.



### Step 1: The First Pick
The algorithm starts by picking the most relevant document.
* **Selected:** **Chunk A** (Similarity 0.95).

### Step 2: Evaluating the Second Pick
The algorithm now calculates the MMR score for the remaining chunks:

* **Chunk B:** While relevance is high (0.92), it is **extremely similar** to Chunk A. The "redundancy penalty" is high, causing its MMR score to drop significantly.
* **Chunk C:** Relevance is moderate (0.85), but it is **highly diverse** compared to Chunk A (Paris vs. Swiss Alps). Because it provides new information, its MMR score remains high.

### Final Result
* **Selected:** **Chunk A** and **Chunk C**.

---

## 4. The Benefit
By using MMR, the LLM receives a much broader context:
1.  **City Tourism:** Paris and the Eiffel Tower.
2.  **Nature/Adventure:** The Swiss Alps and skiing.

**Outcome:** The generated answer will be more comprehensive and helpful to the user, rather than just repeating facts about Paris.

## 5. LangChain Implementation

In LangChain, you can convert any compatible vector store into an MMR retriever using the `search_type="mmr"` argument.

```python
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# Initialize vector store
vectorstore = Chroma(
    persist_directory="./my_db", 
    embedding_function=OpenAIEmbeddings()
)

# Create the MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        'k': 4,              # Final number of chunks to return
        'fetch_k': 20,       # Initial pool to consider for diversity
        'lambda_mult': 0.5   # The balance between relevance (1.0) and diversity (0.0)
    }
)
``` 




# When to Use Maximal Marginal Relevance (MMR)

In LangChain, switching your retriever from the default `similarity` search to `mmr` is highly beneficial in specific scenarios where data variety matters more than pure mathematical proximity.

---

## 1. Key Use Cases for MMR

You should switch to **MMR** if your RAG pipeline faces the following challenges:

### A. Your Data is Repetitive
If your vector store contains multiple versions of the same document, overlapping transcripts, or many similar news articles, a standard search will fill your context window with the same facts repeated in different words.
* **Example:** You have five different support tickets all describing the same "login error." MMR will pick one and then look for other, different types of issues to give the LLM a broader perspective.

### B. The LLM is "Hallucinating" or Getting Confused
When an LLM is given three slightly different versions of the same fact (e.g., different dates for the same event in three different chunks), it often struggles to prioritize the information. This conflict can lead to hallucinations or incoherent answers.
* **MMR Solution:** By providing distinct, non-overlapping facts, you give the LLM a clearer "source of truth" to work with.

### C. You Want Maximum "Coverage"
If a user asks a broad or multi-part question, you need the retriever to explore different facets of the topic rather than digging deep into just one.
* **The "Pros and Cons" Example:** If a user asks, "What are the pros and cons of electric cars?", a standard similarity search might find five different "pro" paragraphs because they all match the "electric car" keywords. MMR forces the retriever to look for the "cons" to maintain diversity in the results.

---

## 2. Summary Comparison

| Feature | Standard Similarity | MMR (Diversity Search) |
| :--- | :--- | :--- |
| **Goal** | Find the absolute closest matches. | Find relevant but unique matches. |
| **Best For** | Fact-checking a single specific point. | Summarization and broad inquiries. |
| **Risk** | Redundancy and wasted context. | May pick a slightly less relevant chunk. |

---

## 3. Implementation Tip
In LangChain, you can quickly test if MMR improves your specific use case by adjusting the `lambda_mult` parameter. 
* Use **0.5** for a balanced approach.
* Use **0.2** if you want to force high diversity (very different chunks).

# Mathematical Walkthrough: MMR with $\lambda = 0.7$

This document breaks down the step-by-step calculation of the **Maximal Marginal Relevance (MMR)** score using a sample dataset.

---

## 1. Setup & Data
**Query ($Q$):** "What are the best vacation spots in Europe?"
**Goal:** Select the top $k=2$ documents from a pool of 4 candidates.
**Lambda ($\lambda$):** 0.7 (70% weight on relevance, 30% weight on diversity).

### Candidate Pool ($fetch\_k$)
| ID | Topic | Sim to Query ($Sim_Q$) | Sim to $D1$ ($Sim_S$) |
| :--- | :--- | :--- | :--- |
| **$D1$** | Paris (General) | 0.95 | - |
| **$D2$** | Paris (Eiffel Tower) | 0.92 | 0.98 |
| **$D3$** | Swiss Alps | 0.85 | 0.30 |
| **$D4$** | Rome (History) | 0.80 | 0.45 |

---

## 2. The Formula
$$\text{Score} = [\lambda \cdot \text{Sim}(D_i, Q)] - [(1 - \lambda) \cdot \max(\text{Sim}(D_i, \text{Selected}))]$$

With $\lambda = 0.7$, the equation becomes:
$$\text{Score} = [0.7 \cdot \text{Sim}(D_i, Q)] - [0.3 \cdot \max(\text{Sim}(D_i, \text{Selected}))]$$

---

## 3. Calculation Steps

### Step 1: Selection of the First Document
The selected set ($S$) is empty. The algorithm simply picks the document with the highest $Sim_Q$.
* **Winner:** **$D1$ (Paris)**
* **Selected Set ($S$):** $\{D1\}$

### Step 2: Selection of the Second Document
We evaluate the remaining candidates ($D2, D3, D4$) against the already selected **$D1$**.



#### Candidate $D2$ (Paris/Eiffel)
* **Relevance Part:** $0.7 \times 0.92 = 0.644$
* **Penalty Part:** $0.3 \times 0.98 = 0.294$
* **Total MMR Score:** $0.644 - 0.294 = \mathbf{0.350}$

#### Candidate $D3$ (Swiss Alps)
* **Relevance Part:** $0.7 \times 0.85 = 0.595$
* **Penalty Part:** $0.3 \times 0.30 = 0.090$
* **Total MMR Score:** $0.595 - 0.090 = \mathbf{0.505}$

#### Candidate $D4$ (Rome)
* **Relevance Part:** $0.7 \times 0.80 = 0.560$
* **Penalty Part:** $0.3 \times 0.45 = 0.135$
* **Total MMR Score:** $0.560 - 0.135 = \mathbf{0.425}$

---

## 4. Final Comparison & Selection

| Candidate | Relevance Score | Diversity Penalty | Final MMR Score |
| :--- | :--- | :--- | :--- |
| $D2$ | 0.644 | 0.294 | 0.350 |
| **$D3$** | **0.595** | **0.090** | **0.505 (WINNER)** |
| $D4$ | 0.560 | 0.135 | 0.425 |

**Result:** The retriever selects **$D1$** and **$D3$**.

### Conclusion
Even though $D2$ had a higher raw similarity to the query (0.92) than $D3$ (0.85), it lost its position because it was **98% similar** to a document already chosen. MMR successfully pivoted to $D3$ to provide a more diverse and informative context for the LLM.

## Example:

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model  # Not working properly so importing ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

from langchain_groq import ChatGroq
 

In [2]:
import langchain
import inspect
print("langchain imported from:", inspect.getfile(langchain))
print("langchain version:", getattr(langchain, "__version__", "NO VERSION FOUND"))

langchain imported from: d:\krishna-codejournal\kcj-langchain-rag-starter\.venv\Lib\site-packages\langchain\__init__.py
langchain version: 1.2.3


In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
 
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found. Check your .env file."


In [4]:
# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [5]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)


In [6]:
### Step 3: Create MMR Retirever
retriever=vectorstore.as_retriever(
    search_type="mmr",
    # search_kwargs={"k":3}
    search_kwargs={"k": 3, "fetch_k": 15, "lambda_mult": 0.7}
)

In [10]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

# llm=init_chat_model("groq:gemma2-9b-it")
# llm = ChatGroq(model="gemma2-9b-it", temperature=0)

llm = init_chat_model("groq:meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)


In [12]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [13]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents and memory in the following ways:

1. **Agents**: LangChain allows Large Language Models (LLMs) to act as agents that can decide which tool to use and in what order during a task. Agents can utilize various tools such as calculators, search APIs, or custom functions based on the instructions they receive.

2. **Memory**: LangChain supports two types of memory:
   - **Conversational Memory**: It is supported using `ConversationBufferMemory`, which helps models retain previous interactions, making multi-turn conversations more coherent.
   - **Summarization Memory**: It is supported using `ConversationSummaryMemory`, which provides a summary of the conversation.

By providing these functionalities, LangChain simplifies the development of complex applications using LLMs.


In [15]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='593ad215-50c3-4b55-8a13-635910eabd7d', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='49174806-5fb5-493f-a123-b9a6cff02c63', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.'),
  Document(id='09b354a6-ff0e-44ba-b499-624ea589f1ac', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using larg

# When NOT to Use Maximal Marginal Relevance (MMR)

Maximal Marginal Relevance (MMR) is a powerful retrieval strategy in RAG systems for balancing **relevance** and **diversity**.  
However, MMR is **not always the best choice**. Below are scenarios where you should consider **skipping MMR** and why.

---

## 🚫 When Not to Use MMR

### 1. Extremely Short Context Window
**Scenario:**  
You can only pass 1–2 chunks to the LLM due to a limited token budget.

**Why skip MMR:**  
MMR’s benefit comes from selecting *multiple* documents while reducing redundancy.  
With a very short context window, you usually want the **single most relevant chunk**, not diversity.

**Better choice:**  
- Top-1 similarity search

---

### 2. You Need Precision Only
**Scenario:**  
The task requires exact or pinpoint answers.

**Examples:**
- “What is ORA-00942?”
- “Which table stores customer balance?”
- “Exact mapping logic for column X”

**Why skip MMR:**  
MMR intentionally trades a bit of relevance for diversity.  
If your goal is **precision over coverage**, this tradeoff can hurt accuracy.

**Better choice:**  
- Pure similarity search  
- Metadata filtering  
- Score thresholds  
- Reranking

---

### 3. Documents Are Already Diverse
**Scenario:**  
Your corpus is naturally diverse, such as:
- One chunk per report
- One chunk per domain/module
- Multiple independent sources

**Why skip MMR:**  
MMR’s main job is to remove redundancy.  
If redundancy is already low, enforcing diversity adds unnecessary overhead.

**Better choice:**  
- Standard similarity retrieval

---

### 4. You’re Already Reranking with an LLM or Cross-Encoder
**Scenario:**  
You already use:
- LLM-based reranking
- Cross-encoder rerankers
- Post-retrieval filtering

**Why skip MMR:**  
Redundancy is already handled downstream during reranking.

**Important nuance:**  
- MMR works **before** LLM invocation (cheaper)
- Reranking works **after** retrieval (more expensive)

In mature systems, both may coexist:
